# Cell Type Annotation and Phenotyping

This notebook performs cell type identification and annotation using:
- Marker gene expression
- Automated annotation tools
- Manual curation
- scVI-tools for probabilistic modeling

**Input:** Preprocessed AnnData object from 01_preprocessing.ipynb

**Output:** Annotated AnnData with cell type labels

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
# === XENIUM_TISSUE dispatch ===
import os, sys, importlib
from pathlib import Path as _Path
_repo = _Path(_os.environ.get("XENIUM_ANALYSIS_ROOT", _Path.cwd().parent)).resolve()
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
_TISSUE = os.environ.get("XENIUM_TISSUE", "pancreas")
_markers = importlib.import_module(f"scripts.tissue_markers.{_TISSUE}")
print(f"[tissue dispatch] XENIUM_TISSUE={_TISSUE}")
# === end XENIUM_TISSUE dispatch ===
warnings.filterwarnings('ignore')

# Rapids GPU-accelerated single-cell ops (drop-in for sc.pp.scale/pca/neighbors,
# sc.tl.score_genes/leiden). UMAP intentionally stays on CPU per CLAUDE.md
# (rapids/cuML UMAP produced extreme outliers on this dataset previously).
import rapids_singlecell as rsc
import cupy as cp

# Enable TF32 / Tensor Cores on B200 (compute capability 10.0).
# Free ~30-50% speedup on FP32 matmul during scVI training.
if torch.cuda.is_available():
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)} (TF32 enabled, "
          f"matmul_precision={torch.get_float32_matmul_precision()})")

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white', frameon=False)

print(f"Scanpy version: {sc.__version__}")
print(f"scVI version: {scvi.__version__}")
print(f"rapids_singlecell version: {rsc.__version__}")


## 1. Load Preprocessed Data

In [ ]:
import os
from pathlib import Path
import numpy as np
import spatialdata as sd

SAMPLE_ID   = os.environ.get("XENIUM_SAMPLE_ID", "0041323")
DATA_DIR    = Path(os.environ.get("XENIUM_OUTPUT_DIR",  f"../data/processed/{SAMPLE_ID}"))
RAW_DIR     = Path(os.environ.get("XENIUM_RAW_DIR", "../data/raw"))
ZARR_PATH   = RAW_DIR / f"{SAMPLE_ID}.zarr"
OUTPUT_DIR  = DATA_DIR
FIGURES_DIR = Path(os.environ.get("XENIUM_FIGURES_DIR", f"../figures/{SAMPLE_ID}/02_phenotyping"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

prep = DATA_DIR / f"{SAMPLE_ID}_preprocessed.h5ad"

if prep.exists():
    print(f"Sample:    {SAMPLE_ID}")
    print(f"Loading preprocessed: {prep}")
    adata = sc.read_h5ad(prep)
    print(f"Loaded shape: {adata.shape}")
    if "leiden_1.5" in adata.obs.columns:
        print(f"Available clusters (leiden_1.5): {adata.obs['leiden_1.5'].nunique()}")
else:
    print(f"Sample:    {SAMPLE_ID}  (standalone — no preprocessed.h5ad)")
    print(f"Loading zarr: {ZARR_PATH}")
    if not ZARR_PATH.exists():
        raise FileNotFoundError(
            f"Neither {prep} nor {ZARR_PATH} exists. Run 00_ingest_xenium.ipynb first."
        )
    sdata = sd.read_zarr(ZARR_PATH)
    adata = sdata.tables["table"]

    # Drop Xenium control probes BEFORE QC
    control_mask = adata.var_names.str.match(
        r"^(NegControlProbe_|UnassignedCodeword_|NegControlCodeword_|antisense_|BLANK_)",
        case=False,
    )
    if control_mask.any():
        print(f"  dropping {int(control_mask.sum())} control probes")
        adata = adata[:, ~control_mask].copy()

    # QC metrics + filter (10x 5k tutorial alignment, same thresholds as 01).
    # Upper bound is DENSITY-based (counts / cell_area > q98) not absolute counts —
    # in FFPE Xenium, large healthy cells (acinar, ductal) have high counts and an
    # absolute cap discards them. See 01_preprocessing_v2.ipynb cell 14 comment.
    sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
    print(f"  before filter: {adata.shape}")
    sc.pp.filter_cells(adata, min_genes=20)
    sc.pp.filter_cells(adata, min_counts=50)
    density = (
        adata.obs["total_counts"].astype(np.float64)
        / np.maximum(adata.obs["cell_area"].astype(np.float64), 1.0)
    )
    density_cutoff = float(np.quantile(density, 0.98))
    print(f"  density cutoff (q98 of counts/area) = {density_cutoff:.3f}")
    adata = adata[(density <= density_cutoff).to_numpy()].copy()
    sc.pp.filter_genes(adata, min_cells=100)
    print(f"  after filter:  {adata.shape}")

    # Preserve raw counts BEFORE any normalization. scVI in cell 17 reads layer='counts'.
    adata.layers["counts"] = adata.X.copy()
    print(f"  counts layer stored")
    print(f"Loaded shape: {adata.shape}")


## 2. Find Cluster Markers

In [ ]:
# Always ensure lognorm + log-normalized .X for downstream score_genes.
# rank_genes_groups (marker discovery by leiden_1.5) only runs if 01_preprocessing
# already produced leiden_1.5 clusters. Without that, scoring proceeds via the
# panel-aware marker dict and we skip the per-cluster DE diagnostic.
if "lognorm" not in adata.layers:
    adata.X = adata.layers["counts"].copy()
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    adata.layers["lognorm"] = adata.X.copy()
else:
    adata.X = adata.layers["lognorm"].copy()

if "leiden_1.5" in adata.obs.columns:
    print("leiden_1.5 present — running rank_genes_groups for marker discovery")
    n_per_group = 1000
    np.random.seed(42)
    indices = []
    for group in adata.obs["leiden_1.5"].unique():
        group_idx = adata.obs.index[adata.obs["leiden_1.5"] == group]
        indices.extend(np.random.choice(group_idx, size=min(n_per_group, len(group_idx)), replace=False))
    adata_de = adata[indices].copy()
    sc.tl.rank_genes_groups(
        adata_de,
        groupby="leiden_1.5",
        method="wilcoxon",
        layer="lognorm",
        key_added="rank_genes_leiden",
        use_raw=False,
    )
    adata.uns["rank_genes_leiden"] = adata_de.uns["rank_genes_leiden"]
    sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False, key="rank_genes_leiden")
    plt.savefig(FIGURES_DIR / "marker_genes_by_cluster.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("Marker gene analysis completed")
else:
    print("[skipped] leiden_1.5 not in adata.obs — skipping rank_genes diagnostic.")
    print("          scVI in cell 17 will compute leiden_scvi for downstream consensus.")


In [ ]:
if "rank_genes_leiden" in adata.uns:
    result = adata.uns["rank_genes_leiden"]
    groups = result["names"].dtype.names
    marker_df = pd.DataFrame({
        group + "_" + key: result[key][group]
        for group in groups for key in ["names", "scores"]
    })
    marker_df.to_csv(OUTPUT_DIR / f"{SAMPLE_ID}_marker_genes.csv")
    print(f"Marker genes saved to {OUTPUT_DIR / f'{SAMPLE_ID}_marker_genes.csv'}")
    print("\nTop 5 marker genes per cluster:")
    for cluster in groups[:5]:
        print(f"\nCluster {cluster}:")
        print(marker_df[f"{cluster}_names"].head().tolist())
else:
    print("[skipped] no rank_genes_leiden — see cell 5.")


## 3. Define Cell Type Markers

Define known marker genes for major cell types (customize based on tissue type)

In [6]:
adata

AnnData object with n_obs × n_vars = 521984 × 5081
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_genes', 'n_counts', 'leiden_1.5'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'mean', 'std', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'dendrogram_leiden_1.5', 'hvg', 'leiden_1.5', 'leiden_1.5_centrality_scores', 'leiden_1.5_colors', 'leiden_1.5_nhood_enrichment', 'log1p',

In [ ]:
if "rank_genes_leiden" in adata.uns:
    def _add_group_separators(dp_or_mp, group_dict_or_n_per_group, n_groups=None,
                              color="black"):
        """Draw vertical lines BETWEEN gene groups on a scanpy DotPlot/MatrixPlot.
        Pass the object returned by sc.pl.* with return_fig=True. We force-render
        via make_figure() if needed, then add axvlines to the mainplot_ax at the
        correct boundary positions (scanpy places genes at 0.5, 1.5, 2.5, ...
        so boundaries between groups are at integer cumulative gene counts).
        """
        if dp_or_mp.ax_dict is None:
            dp_or_mp.make_figure()
        main_ax = dp_or_mp.ax_dict.get("mainplot_ax")
        if main_ax is None:
            return
        if isinstance(group_dict_or_n_per_group, dict):
            cum = 0
            items = list(group_dict_or_n_per_group.items())
            for i, (_, genes) in enumerate(items):
                cum += len(genes)
                if i < len(items) - 1:
                    main_ax.axvline(cum, color=color, linewidth=1.6, alpha=0.95, zorder=10)
        else:
            n_per = int(group_dict_or_n_per_group)
            for k in range(1, int(n_groups)):
                main_ax.axvline(k * n_per, color=color, linewidth=1.6, alpha=0.95, zorder=10)
    hm = sc.pl.rank_genes_groups_heatmap(
        adata, n_genes=5, key="rank_genes_leiden", show_gene_labels=True,
        figsize=(28, 28), cmap="viridis", vmin=-1, vmax=3,
        layer="lognorm", swap_axes=True, var_group_rotation=90, show=False,
        return_fig=True,
    )
    n_clusters = adata.obs["leiden_1.5"].astype("category").cat.categories.size if "leiden_1.5" in adata.obs.columns else 0
    if n_clusters >= 2 and hasattr(hm, "ax_dict"):
        _add_group_separators(hm, 5, n_groups=n_clusters, color="white")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "marker_genes_heatmap.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("[skipped] marker_genes_heatmap — no rank_genes_leiden.")


In [ ]:
# Panel-aware pancreas markers for the T1D-focused hAtlas_v1.1+100 panel.
# Absent from panel: INS, GCG, SST, PPY, IRX2, PRSS1/2, CTRB1/2, CELA1/2A/3A,
# REG1A/3A. Beta uses surrogates; Alpha/Delta use multi-marker panels.
#
# RARE TYPES (Epsilon, Gamma) are NOT scored via score_genes argmax —
# they're identified post-hoc in scripts/refine_rare_celltypes.py via
# multi-criteria rules (e.g. pan-endocrine + AQP3+ETV1 + NOT Beta/Delta
# = Gamma). Score-genes argmax with weak/single-gene panels mislabeled
# 150K cells in 0041326 as Epsilon under the previous design (the bug
# that triggered this rebuild).
#
# Acinar reduces to AMY1A + AMY2A + AMY2B; the anti-acinar override
# in cell 15 routes mis-Acinar calls based on stronger alt-lineage
# markers.
pancreas_markers = _markers.MARKER_DICT
# Low-confidence types (panel coverage thin). Used to downgrade ambiguous
# assignments via the post-hoc panel-coverage check in
# scripts/refine_rare_celltypes.py. Epsilon removed (no longer score_genes-target).
LOW_CONFIDENCE_TYPES = set(_markers.LOW_CONFIDENCE_TYPES)
# Apply the panel-audit exclusion mask. Set by scripts/finalize_panel.py.
# Drops 100 non-sex tissue-restricted genes + 6 sex-chromosome genes from
# the per-type panels.
if 'panel_for_scoring' in adata.var.columns:
    panel_keep = set(adata.var_names[adata.var['panel_for_scoring'].astype(bool)])
    print(f"panel_for_scoring: keeping {len(panel_keep):,} of {adata.n_vars:,} genes")
else:
    print("WARNING: var['panel_for_scoring'] not set — running with full panel. "
          "Run scripts/finalize_panel.py before re-running.")
    panel_keep = set(adata.var_names)

available_markers = {}
for ct, genes in pancreas_markers.items():
    kept = [g for g in genes if g in adata.var_names and g in panel_keep]
    if not kept:
        print(f"  {ct}: NO markers on panel; dropped from scoring")
        continue
    if len(kept) < 2:
        # CLAUDE.md decision #12: panels with <2 genes too noisy for argmax.
        # The single-gene panel was the Epsilon bug.
        print(f"  {ct}: only {len(kept)} marker(s) ({kept}); dropped from scoring "
              f"to prevent single-gene mislabel bug")
        continue
    available_markers[ct] = kept
    if len(kept) < 3:
        print(f"  WARNING: {ct} has only {len(kept)} markers ({kept}); flagging "
              f"as LOW_CONFIDENCE_TYPES even if not currently in the set")
        LOW_CONFIDENCE_TYPES.add(ct)

print(f"\nCell types scored via score_genes argmax: {list(available_markers.keys())}")
print(f"LOW_CONFIDENCE_TYPES (downgrade-candidates after argmax): {sorted(LOW_CONFIDENCE_TYPES)}")
print("Epsilon + Gamma identified post-hoc via rule-based multi-criteria — "
      "see scripts/refine_rare_celltypes.py.")


In [ ]:
if "leiden_1.5" in adata.obs.columns:
    def _add_group_separators(dp_or_mp, group_dict_or_n_per_group, n_groups=None,
                              color="black"):
        """Draw vertical lines BETWEEN gene groups on a scanpy DotPlot/MatrixPlot.
        Pass the object returned by sc.pl.* with return_fig=True. We force-render
        via make_figure() if needed, then add axvlines to the mainplot_ax at the
        correct boundary positions (scanpy places genes at 0.5, 1.5, 2.5, ...
        so boundaries between groups are at integer cumulative gene counts).
        """
        if dp_or_mp.ax_dict is None:
            dp_or_mp.make_figure()
        main_ax = dp_or_mp.ax_dict.get("mainplot_ax")
        if main_ax is None:
            return
        if isinstance(group_dict_or_n_per_group, dict):
            cum = 0
            items = list(group_dict_or_n_per_group.items())
            for i, (_, genes) in enumerate(items):
                cum += len(genes)
                if i < len(items) - 1:
                    main_ax.axvline(cum, color=color, linewidth=1.6, alpha=0.95, zorder=10)
        else:
            n_per = int(group_dict_or_n_per_group)
            for k in range(1, int(n_groups)):
                main_ax.axvline(k * n_per, color=color, linewidth=1.6, alpha=0.95, zorder=10)
    dp = sc.pl.dotplot(
        adata, available_markers, groupby="leiden_1.5",
        dendrogram=True, cmap="Reds", figsize=(18, 10),
        vmin=0.0, vmax=1.0, standard_scale="var",
        swap_axes=False, var_group_rotation=90, show=False,
        return_fig=True,
        mean_only_expressed=True,
    )
    _add_group_separators(dp, available_markers, color="black")
    plt.savefig(FIGURES_DIR / "dotplot_markers_by_cluster.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("[skipped] leiden_1.5 dotplot — will plot by leiden_scvi in cell 18 after scVI runs.")

## 5. Score Cell Types

In [ ]:
# Score cells for each cell type using the log-normalized layer.
# Plot deferred to AFTER cell 17 (where scVI computes X_umap). Plotting here
# would render scores on the stale Xenium-builtin UMAP from sd.read_zarr.
for cell_type, genes in available_markers.items():
    score_name = f"{cell_type}_score"
    sc.tl.score_genes(adata, genes, score_name=score_name, use_raw=False)
    print(f"Scored {cell_type}")

print("Score columns ready; UMAP plot deferred until cell 17 finishes scVI UMAP.")


## 6. Assign Cell Types Based on Scores

In [ ]:
# Assign cell types by highest score; flag low-confidence assignments for types
# whose panel coverage is thin. Threshold is per-dataset
# (median - 1*MAD on celltype_confidence).
#
# After initial argmax assignment, run an "anti-dominant-type override"
# pass: the panel has only AMY1A as the acinar marker (pancreas) or DP
# thymocytes share most T-lineage markers with SP4/SP8 (thymus), so any
# cell with detectable bulk-type markers can argmax-out as that type even
# when it's actually expressing strong, specific markers of another lineage
# (Ductal KRT19/CFTR, Endothelial PECAM1/CDH5, immune CD3D/CD68, etc.).
# For each dominant-type-called cell, we check expression of curated SPECIFIC
# alt markers in the lognorm layer; if any alt set's mean exceeds the median
# across cells already confidently labeled that type, we reassign.
#
# 2026-05-22: celltype_confidence now reports the LINEAGE margin (top
# lineage score minus second lineage score), aggregated via
# _markers.LINEAGE_GROUPS. The previous top-2 cell-type margin reported
# near-zero confidence for any clear thymocyte (DP/SP4/SP8 share
# CD3E/CD3G/TRAC/TRBC2 → all three score similarly → margin → 0),
# producing the "vast majority low-confidence" appearance even when the
# lineage call was unambiguous. The legacy top-2-cell-type margin is
# preserved as celltype_confidence_margin_legacy for audit purposes.
# Within-lineage margin is preserved as subtype_confidence.

from scipy.sparse import issparse as _issparse

# Curated SPECIFIC anti-dominant-type marker sets (must NOT include genes that
# bleed across cell types). Each set is a small high-confidence panel.
ANTI_ACINAR_MARKERS = _markers.ANTI_OVERRIDE_MARKERS
def _per_cell_marker_mean(adata_, gene_list, layer="lognorm"):
    """Per-cell mean of `gene_list` from layers[layer]. Returns 1D np array."""
    present = [g for g in gene_list if g in adata_.var_names]
    if not present:
        return None
    X = adata_[:, present].layers[layer]
    if _issparse(X):
        return np.asarray(X.mean(axis=1)).ravel()
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = [f"{ct}_score" for ct in available_markers.keys()]

if score_cols:
    scores_df = adata.obs[score_cols]
    adata.obs["predicted_celltype"] = scores_df.idxmax(axis=1).str.replace("_score", "")
    # Preserve the legacy top-2-cell-type margin for audit purposes.
    top2_scores = np.sort(scores_df.values, axis=1)[:, -2:]
    adata.obs["celltype_confidence_margin_legacy"] = top2_scores[:, 1] - top2_scores[:, 0]
    # celltype_confidence below is overwritten with the lineage margin after
    # the anti-dominant-type override completes — the override may change
    # predicted_celltype, which would shift subtype_confidence semantics.
    adata.obs["celltype_confidence"] = adata.obs["celltype_confidence_margin_legacy"]

    # ---------- Anti-dominant-type override pass ----------
    print(f"\n--- Anti-{_markers.ANTI_OVERRIDE_DOMINANT_TYPE} override ---")
    pred = adata.obs["predicted_celltype"].astype(str).to_numpy()
    n_acinar_initial = int((pred == _markers.ANTI_OVERRIDE_DOMINANT_TYPE).sum())
    print(f"  initial {_markers.ANTI_OVERRIDE_DOMINANT_TYPE} calls: {n_acinar_initial:,}")

    # Pre-compute alt marker means for each candidate alt type AND a baseline
    # threshold (median of that mean across cells confidently called the alt type).
    alt_mean = {}
    alt_thr = {}
    for alt_ct, marker_list in ANTI_ACINAR_MARKERS.items():
        m = _per_cell_marker_mean(adata, marker_list, layer="lognorm")
        if m is None or m.size == 0:
            continue
        alt_mean[alt_ct] = m
        # Threshold: median of `m` over cells initially called `alt_ct`.
        # Fallback: 75th percentile of `m` across all cells (so we don't override on noise).
        ct_initial_mask = (pred == alt_ct)
        if ct_initial_mask.sum() >= 50:
            alt_thr[alt_ct] = float(np.median(m[ct_initial_mask]))
        else:
            alt_thr[alt_ct] = float(np.quantile(m, 0.75))
        print(f"  {alt_ct}: marker_mean threshold = {alt_thr[alt_ct]:.3f} "
              f"(initial cells of this type: {int(ct_initial_mask.sum()):,})")

    # 2026-05-20 — T-lineage protection (thymus): do not flip cells with
    # CD3E or TRAC detected at lognorm >= log1p(1). FFPE neighbor-cell
    # contamination otherwise triggers anti-override on confirmed thymocytes.
    # For pancreas this is a near-no-op (CD3E/TRAC very rare; ANTI_OVERRIDE_DOMINANT_TYPE
    # = Acinar, so t_lineage protection on a tiny T-cell population is harmless).
    import numpy as _np
    _log_present = float(_np.log1p(1.0))
    _cd3e_vec = _per_cell_marker_mean(adata, ["CD3E"], layer="lognorm")
    _trac_vec = _per_cell_marker_mean(adata, ["TRAC"], layer="lognorm")
    if _cd3e_vec is not None and _trac_vec is not None:
        t_lineage = (_cd3e_vec >= _log_present) | (_trac_vec >= _log_present)
    elif _cd3e_vec is not None:
        t_lineage = _cd3e_vec >= _log_present
    elif _trac_vec is not None:
        t_lineage = _trac_vec >= _log_present
    else:
        t_lineage = _np.zeros(adata.n_obs, dtype=bool)
    print(f"  T-lineage protect: {int(t_lineage.sum()):,} cells immune to anti-override")

    # For each dominant-type-called cell, find the alt type with the largest
    # (alt_mean - alt_thr) margin (only positives count). Reassign if any positive.
    dominant_mask = (pred == _markers.ANTI_OVERRIDE_DOMINANT_TYPE)
    if dominant_mask.any() and alt_mean:
        # Stack alt margins as a (n_alt_types, n_cells) array
        alt_names = list(alt_mean.keys())
        margins = np.stack([alt_mean[a] - alt_thr[a] for a in alt_names], axis=0)  # (T, N)
        # For non-dominant cells, mask margins to -inf so they aren't affected
        margins[:, ~dominant_mask] = -np.inf
        best_alt_idx = margins.argmax(axis=0)
        best_alt_margin = margins[best_alt_idx, np.arange(len(pred))]
        # Reassign dominant cells whose best alt margin > 0
        reassign = dominant_mask & (best_alt_margin > 0) & ~t_lineage
        n_reassigned = int(reassign.sum())
        print(f"  {_markers.ANTI_OVERRIDE_DOMINANT_TYPE} cells with positive alt margin: {n_reassigned:,} ({100*n_reassigned/max(n_acinar_initial,1):.1f}%)")
        if n_reassigned > 0:
            new_labels = np.array(alt_names)[best_alt_idx]
            pred[reassign] = new_labels[reassign]
            # Per-type breakdown
            unique, counts = np.unique(new_labels[reassign], return_counts=True)
            for u, c in sorted(zip(unique, counts), key=lambda kv: -kv[1]):
                print(f"    -> {u}: +{int(c):,}")
        adata.obs["predicted_celltype"] = pd.Categorical(pred)

    # ---------- Lineage-margin celltype_confidence (2026-05-22 rewrite) ----------
    # Compute per-lineage scores (mean of member-celltype score columns) and
    # report lineage_confidence = top lineage score - second lineage score.
    # Within-lineage margin (top within-lineage cell-type score minus second
    # within-lineage) preserved as subtype_confidence.
    if hasattr(_markers, "LINEAGE_GROUPS"):
        print("\n--- Computing lineage-margin celltype_confidence ---")
        lineage_groups = _markers.LINEAGE_GROUPS
        scored_types = set(available_markers.keys())
        # Restrict each lineage to its scored members. Lineages with no scored
        # members (e.g. only post-hoc rule-based types) are dropped silently.
        lineage_members = {
            lin: sorted(scored_types & set(types))
            for lin, types in lineage_groups.items()
        }
        lineage_members = {lin: ms for lin, ms in lineage_members.items() if ms}
        print(f"  lineages scored: {sorted(lineage_members)}")
        # Per-cell lineage score = mean of member celltype scores.
        lineage_score_mat = np.zeros((adata.n_obs, len(lineage_members)),
                                     dtype=np.float32)
        lineage_names = list(lineage_members.keys())
        scores_np = scores_df.values  # (N, n_celltypes)
        col_idx = {col: i for i, col in enumerate(score_cols)}
        for j, lin in enumerate(lineage_names):
            member_cols = [col_idx[f"{ct}_score"] for ct in lineage_members[lin]]
            lineage_score_mat[:, j] = scores_np[:, member_cols].mean(axis=1)
        # Top-2 lineage margin.
        top2_lin = np.sort(lineage_score_mat, axis=1)[:, -2:]
        lineage_confidence = (top2_lin[:, 1] - top2_lin[:, 0]).astype(np.float32)
        predicted_lineage_idx = lineage_score_mat.argmax(axis=1)
        predicted_lineage = np.array(lineage_names)[predicted_lineage_idx]
        # Within-lineage subtype margin: for each cell, mask scores not in its
        # predicted lineage to -inf, then take top-2 margin.
        celltype_to_lineage_idx = np.full(len(score_cols), -1, dtype=np.int32)
        for j, lin in enumerate(lineage_names):
            for ct in lineage_members[lin]:
                celltype_to_lineage_idx[col_idx[f"{ct}_score"]] = j
        # If a celltype isn't in any LINEAGE_GROUPS entry leave it as -1; it
        # contributes to neither subtype margin (will compare against itself).
        in_pred_lineage = (celltype_to_lineage_idx[np.newaxis, :]
                           == predicted_lineage_idx[:, np.newaxis])  # (N, K)
        masked = np.where(in_pred_lineage, scores_np, -np.inf)
        # When a lineage has only one member, second-top is -inf → margin inf.
        # Treat singleton lineages as "no subtype ambiguity" (subtype_conf=NaN).
        sorted_masked = np.sort(masked, axis=1)
        top_sub = sorted_masked[:, -1]
        second_sub = sorted_masked[:, -2]
        subtype_confidence = np.where(np.isinf(second_sub),
                                      np.nan,
                                      (top_sub - second_sub)).astype(np.float32)
        adata.obs["predicted_lineage"] = pd.Categorical(predicted_lineage,
                                                         categories=lineage_names)
        adata.obs["lineage_confidence"] = lineage_confidence
        adata.obs["subtype_confidence"] = subtype_confidence
        adata.obs["celltype_confidence"] = lineage_confidence  # overwrite
        print(f"  predicted_lineage: {sorted(set(predicted_lineage))}")
        print(f"  lineage_confidence: median={float(np.median(lineage_confidence)):.4f}, "
              f"p90={float(np.quantile(lineage_confidence, 0.90)):.4f}")
        print(f"  subtype_confidence: "
              f"median={float(np.nanmedian(subtype_confidence)):.4f}, "
              f"nan_count={int(np.isnan(subtype_confidence).sum()):,}")
    else:
        print(f"\n[skipped] _markers.LINEAGE_GROUPS not defined for tissue "
              f"'{_TISSUE}'; celltype_confidence remains top-2 cell-type margin.")

    # ---------- Indeterminate flagging (post-override) ----------
    print("\n--- Confidence-based Indeterminate flagging ---")
    conf = adata.obs["celltype_confidence"].to_numpy()
    pred = adata.obs["predicted_celltype"].astype(str).to_numpy()
    low_conf = pred.copy()
    for ct in LOW_CONFIDENCE_TYPES:
        mask = pred == ct
        if mask.sum() < 10:
            continue
        c = conf[mask]
        med = np.median(c)
        mad = np.median(np.abs(c - med)) + 1e-12
        thr = med - mad
        is_indet = mask & (conf < thr)
        bucket = _markers.INDETERMINATE_BUCKETS.get(ct, f"Indeterminate_{ct}")
        low_conf[is_indet] = bucket
        print(f"  {ct}: flagged {int(is_indet.sum())}/{int(mask.sum())} as {bucket} (thr conf<{thr:.3f})")
    adata.obs["predicted_celltype"] = pd.Categorical(low_conf)

    print("\nCell type distribution (post-anti-override + confidence-gate):")
    print(adata.obs["predicted_celltype"].value_counts())

    # Plot deferred to cell 17 (so it lands on the scVI UMAP, not the stale Xenium UMAP).

## 7. scVI-tools for Advanced Analysis (Optional)

Use scVI for probabilistic modeling and better integration

In [ ]:
# scVI on B200 GPU (TF32 enabled in cell 1) -> rapids_singlecell neighbors
# (cuML kNN, GPU) -> scanpy CPU UMAP -> scanpy CPU leiden (igraph flavor).
# Library-size effect is modelled internally by scVI's NB likelihood; passing
# total_counts as a covariate here would double-regress it. cell_area (FFPE
# segmentation size) IS a valid per-cell nuisance covariate to hand scVI.
# UMAP intentionally stays on CPU: rapids/cuML UMAP produced extreme outliers
# on this dataset previously (see CLAUDE.md). Leiden on CPU because cugraph
# leiden currently has a dask-cuda<->dask version drift.
import torch

# Train scVI on the panel_for_embedding subset only — drops Y-chromosome
# genes (KDM5D, DDX3Y et al.) so the donor-sex difference between samples
# (0041326 male, 0041323 female) doesn't dominate the latent. Also drops
# the 100 tissue-restricted genes flagged in panel_audit.csv that have
# no biological role in pancreas / LN / vessel context.
# donor_sex obs-column is set by scripts/finalize_panel.py from
# KDM5D + DDX3Y detection rates.
if 'panel_for_embedding' in adata.var.columns:
    embed_mask = adata.var['panel_for_embedding'].astype(bool).to_numpy()
    print(f"scVI training on {int(embed_mask.sum()):,} of {adata.n_vars:,} panel_for_embedding genes")
    adata_scvi = adata[:, embed_mask].copy()
else:
    print("WARNING: var['panel_for_embedding'] not set; training scVI on full panel "
          "(sex-chromosome leak likely). Run scripts/finalize_panel.py first.")
    adata_scvi = adata

cat_covariates = []
if 'donor_sex' in adata_scvi.obs.columns:
    cat_covariates = ['donor_sex']
    print(f"scVI categorical_covariate_keys: {cat_covariates}")

scvi.model.SCVI.setup_anndata(
    adata_scvi,
    layer='counts',
    batch_key=None,
    continuous_covariate_keys=['cell_area'],
    categorical_covariate_keys=cat_covariates if cat_covariates else None,
)
vae = scvi.model.SCVI(adata_scvi, n_layers=2, n_latent=30, gene_likelihood='nb', dropout_rate=0.1)

print("Training scVI model ...")
use_gpu = torch.cuda.is_available()
print(f"  CUDA available: {use_gpu}")
# 2026-05-22: bumped max_epochs 200 -> 400 and early_stopping_patience 20 -> 40
# per HANDOFF 2026-05-21 G2 fallback. Targets 0041534's borderline library-size
# leak (rho=0.353 on X-axis with old config). Early stopping still active, so
# samples that converge fast won't pay the full 400-epoch cost.
vae.train(
    max_epochs=400,
    early_stopping=True,
    early_stopping_patience=40,
    accelerator=('gpu' if use_gpu else 'cpu'),
    devices=1,
    batch_size=4096,
    datasplitter_kwargs={"num_workers": 8, "pin_memory": True, "persistent_workers": True},
)
adata.obsm['X_scvi'] = vae.get_latent_representation()  # row-aligned with adata
del adata_scvi

# Primary embedding: scVI latent -> cosine neighbours -> UMAP -> leiden (igraph flavor).
# Parameters aligned with 10x Xenium 5k tutorial (2026).
print("scVI neighbors/UMAP/leiden (scanpy, CPU, cosine, min_dist=0.05, spread=2.5, igraph) ...")
rsc.pp.neighbors(adata, use_rep='X_scvi', n_neighbors=10, metric='cosine', random_state=0, key_added='scvi')  # GPU
sc.tl.umap(adata, neighbors_key='scvi', min_dist=0.05, spread=2.5, random_state=0)
sc.tl.leiden(
    adata, resolution=1.0, neighbors_key='scvi', key_added='leiden_scvi',
    flavor='igraph', n_iterations=-1, directed=False, random_state=0,
)

# ---- Validation overlay: library size MUST NOT drive the embedding ----
um = adata.obsm['X_umap']
from scipy.stats import spearmanr as _spearmanr
_r1, _ = _spearmanr(um[:, 0], adata.obs['total_counts'])
_r2, _ = _spearmanr(um[:, 1], adata.obs['total_counts'])
print(f"  Spearman(UMAP axis vs total_counts): {_r1:.3f} (x), {_r2:.3f} (y)")
if max(abs(_r1), abs(_r2)) > 0.3:
    print(f"  WARNING: library size still leaks into scVI UMAP (|rho|={max(abs(_r1),abs(_r2)):.2f}>0.3).")
    print(f"  The PCA-fallback UMAP will be checked as a sanity backup.")

# scVI-latent diagnostic (the representation phenotyping actually uses, not the
# gene-space PCA scree which was dominated by the z-scored noise floor). Panel A:
# scree of PCA on X_scvi (cumulative -> 1.0, effective-dim marker). Panel B:
# per-latent-dim usage + the library-size leak gate above. Shared with
# scripts/render_scvi_latent_diagnostic.py so notebook and re-render match.
from utils.analysis_utils import plot_scvi_latent_diagnostic
plot_scvi_latent_diagnostic(adata, FIGURES_DIR / "scvi_latent_diagnostic.png", show=True)

# Color scaling: clip continuous metrics at the 95th percentile so the bulk
# distribution spans the full viridis range (otherwise a few extreme cells
# squash 95%+ of cells into the dark-purple end and the plot looks uniform).
import numpy as _np

# ---- Deferred figures on the scVI UMAP, via shared utils helpers so the
#      notebook matches scripts/rerender_stage02_celltype_figs.py styling
#      (adaptive point size + lineage-subset split). These are provisional
#      (predicted_celltype, Indeterminate hidden); the post-resolve re-render
#      script overwrites them with the resolved `celltype`.
from utils.analysis_utils import umap_dot_size, lineage_subsets_for, render_celltype_subset_panels, render_major_lineage_panel
_sz = umap_dot_size(adata.n_obs)

# umap_qc_overlay.png
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for _col, _ax, _title in [("total_counts", axes[0, 0], "Total counts"),
                          ("n_genes_by_counts", axes[0, 1], "N genes/cell"),
                          ("cell_area", axes[1, 0], "Cell area")]:
    _vmax = float(np.quantile(adata.obs[_col], 0.95))
    sc.pl.umap(adata, color=_col, ax=_ax, show=False, cmap="viridis",
               title=f"{_title} (clipped p95={_vmax:.0f})",
               vmin=float(adata.obs[_col].min()), vmax=_vmax, size=_sz)
sc.pl.umap(adata, color="leiden_scvi", ax=axes[1, 1], show=False, title="leiden_scvi",
           legend_loc="on data", legend_fontsize=11, legend_fontoutline=2, size=_sz)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "umap_qc_overlay.png", dpi=200, bbox_inches="tight")
plt.show()

# celltype_scores_umap.png
_score_cols = [c for c in adata.obs.columns if c.endswith("_score")]
if _score_cols:
    _nc = 3; _nr = (len(_score_cols) + _nc - 1) // _nc
    fig, axes = plt.subplots(_nr, _nc, figsize=(5 * _nc, 4 * _nr))
    axes = np.atleast_1d(axes).flatten()
    for _i, _s in enumerate(_score_cols):
        sc.pl.umap(adata, color=_s, ax=axes[_i], show=False, cmap="viridis",
                   title=_s, size=_sz, frameon=False)
    for _j in range(len(_score_cols), len(axes)):
        axes[_j].axis("off")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "celltype_scores_umap.png", dpi=200, bbox_inches="tight")
    plt.show()

# predicted_celltypes.png — confidence anchor + celltype view. View is
# tissue-dispatched via _markers.PREDICTED_VIEW: "major_lineage" (pancreas —
# one de-crowded ~7-group LINEAGE_MAP panel) or "lineage_subsets" (the 3-panel
# split). Shares utils helpers with scripts/rerender_stage02_celltype_figs.py.
_pc = adata.obs["predicted_celltype"].astype(str)
_a_clean = adata[(~_pc.str.startswith("Indeterminate")).values].copy() \
    if _pc.str.startswith("Indeterminate").any() else adata.copy()
_view = getattr(_markers, "PREDICTED_VIEW", "lineage_subsets")
_subsets = lineage_subsets_for(_TISSUE)
if _view == "major_lineage":
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    if "celltype_confidence" in adata.obs.columns:
        _cf = adata.obs["celltype_confidence"].to_numpy()
        _vx = float(np.quantile(_cf[~np.isnan(_cf)], 0.99))
        sc.pl.umap(_a_clean, color="celltype_confidence", ax=axes[0], show=False,
                   cmap="viridis", size=umap_dot_size(_a_clean.n_obs), frameon=False,
                   vmax=_vx, title="celltype_confidence (lineage margin)")
    else:
        axes[0].axis("off")
    render_major_lineage_panel(_a_clean, axes[1], _markers.LINEAGE_MAP,
                               _markers.PALETTE, color="predicted_celltype",
                               title="Major lineage (marker-based)")
elif _subsets is None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sc.pl.umap(_a_clean, color="predicted_celltype", ax=axes[0], show=False,
               size=umap_dot_size(_a_clean.n_obs), frameon=False,
               title="Marker-based prediction (Indeterminate hidden)")
    if "celltype_confidence" in adata.obs.columns:
        sc.pl.umap(_a_clean, color="celltype_confidence", ax=axes[1], show=False,
                   cmap="viridis", size=umap_dot_size(_a_clean.n_obs), frameon=False,
                   title="celltype_confidence (lineage margin)")
    else:
        axes[1].axis("off")
else:
    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    if "celltype_confidence" in adata.obs.columns:
        _cf = adata.obs["celltype_confidence"].to_numpy()
        _vx = float(np.quantile(_cf[~np.isnan(_cf)], 0.99))
        sc.pl.umap(_a_clean, color="celltype_confidence", ax=axes[0, 0], show=False,
                   cmap="viridis", size=umap_dot_size(_a_clean.n_obs), frameon=False,
                   vmax=_vx, title="celltype_confidence (lineage margin)")
    else:
        axes[0, 0].set_axis_off()
    render_celltype_subset_panels(_a_clean, [axes[0, 1], axes[1, 0], axes[1, 1]],
                                  _subsets[:3], color="predicted_celltype")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "predicted_celltypes.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# scvi_vs_markers.png — leiden_scvi + celltype view of the marker-based
# prediction (Indeterminate hidden). View tissue-dispatched via
# _markers.PREDICTED_VIEW ("major_lineage": one ~7-group LINEAGE_MAP panel;
# else the 3-panel LINEAGE_SUBSETS split). Shares utils helpers with
# scripts/rerender_stage02_celltype_figs.py (re-renders the resolved version
# post-resolve).
from utils.analysis_utils import umap_dot_size, lineage_subsets_for, render_celltype_subset_panels, render_major_lineage_panel
_pc = adata.obs['predicted_celltype'].astype(str)
_mask = ~_pc.str.startswith('Indeterminate')
_a_clean = adata[_mask.values].copy() if (~_mask).any() else adata.copy()
_view = getattr(_markers, 'PREDICTED_VIEW', 'lineage_subsets')
_subsets = lineage_subsets_for(_TISSUE)
if _view == 'major_lineage':
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    sc.pl.umap(_a_clean, color='leiden_scvi', ax=axes[0], show=False,
               size=umap_dot_size(_a_clean.n_obs), frameon=False, legend_loc='on data',
               legend_fontsize=10, legend_fontoutline=2, title='scVI Leiden')
    render_major_lineage_panel(_a_clean, axes[1], _markers.LINEAGE_MAP,
                               _markers.PALETTE, color='predicted_celltype',
                               title='Major lineage (marker-based)')
elif _subsets is None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    _sz = umap_dot_size(_a_clean.n_obs)
    sc.pl.umap(_a_clean, color='leiden_scvi', ax=axes[0], show=False, size=_sz, title='scVI Leiden')
    sc.pl.umap(_a_clean, color='predicted_celltype', ax=axes[1], show=False, size=_sz,
               title='Marker-based prediction (Indeterminate hidden)')
else:
    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    sc.pl.umap(_a_clean, color='leiden_scvi', ax=axes[0, 0], show=False,
               size=umap_dot_size(_a_clean.n_obs), frameon=False, legend_loc='on data',
               legend_fontsize=10, legend_fontoutline=2, title='scVI Leiden')
    render_celltype_subset_panels(_a_clean, [axes[0, 1], axes[1, 0], axes[1, 1]],
                                  _subsets[:3], color='predicted_celltype')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'scvi_vs_markers.png', dpi=300, bbox_inches='tight')
plt.show()
if (~_mask).any():
    print(f"  scvi_vs_markers.png: hid {int((~_mask).sum()):,} Indeterminate cells "
          f"(resolved downstream by scripts/resolve_indeterminate.py)")

## 8. Manual Refinement (if needed)

In [ ]:
# Auto-derive per-cluster consensus cell type from the per-cell `predicted_celltype` (score-based).
# Take the majority label within each leiden_scvi cluster; this gives clean, discretized labels
# without requiring manual inspection of the marker heatmap.
# Override the auto map by populating `manual_override` for individual clusters after reviewing
# figures/.../02_phenotyping/marker_genes_heatmap.png.

auto_annotation = (
    adata.obs.groupby('leiden_scvi', observed=True)['predicted_celltype']
    .agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)

# Per-cluster purity (fraction of cells matching the consensus label) — low purity flags mixed clusters
purity = (
    adata.obs.groupby('leiden_scvi', observed=True)['predicted_celltype']
    .agg(lambda s: s.value_counts(normalize=True).iloc[0])
    .to_dict()
)
cluster_sizes = adata.obs['leiden_scvi'].value_counts().to_dict()

print(f"{'cluster':>8}  {'size':>8}  {'consensus':>14}  purity")
for cl in sorted(auto_annotation, key=lambda k: int(k) if str(k).isdigit() else k):
    print(f"{cl:>8}  {cluster_sizes[cl]:>8}  {auto_annotation[cl]:>14}  {purity[cl]:.2f}")

# Populate this dict to override auto assignments for specific clusters after inspecting markers.
# Keys: leiden_scvi cluster id (str); values: cell type name.
manual_override = {
    # '0': 'Acinar',
    # '7': 'Beta',
}

cluster_annotation = {**auto_annotation, **manual_override}

adata.obs['manual_celltype'] = adata.obs['leiden_scvi'].map(cluster_annotation).astype('category')

# 2026-05-20 — Thymocyte preservation guard: cluster-majority would collapse
# DP/SP4/SP8/DN diversity within mixed cortical thymocyte clusters. Restore
# per-cell predicted_celltype for any cell whose predicted_celltype is a
# thymocyte subtype. For pancreas this is a no-op (no thymocyte names in
# MARKER_DICT).
_THY_SUBTYPES = {"DN_thymocyte", "DP_thymocyte", "SP4_thymocyte",
                  "SP8_thymocyte", "Treg"}
_is_thy = adata.obs['predicted_celltype'].astype(str).isin(_THY_SUBTYPES)
if _is_thy.any():
    _mc = adata.obs['manual_celltype'].astype(str)
    _mc[_is_thy.values] = adata.obs.loc[_is_thy, 'predicted_celltype'].astype(str).values
    adata.obs['manual_celltype'] = pd.Categorical(_mc)
    print(f"Thymocyte preservation: restored {int(_is_thy.sum()):,} cells "
          f"from cluster-majority to per-cell predicted_celltype "
          f"({adata.obs['manual_celltype'].nunique()} unique types)")

print(f"\nAssigned {adata.obs['manual_celltype'].nunique()} cell types across "
      f"{adata.obs['leiden_scvi'].nunique()} clusters")

# Three-panel UMAP: general lineage | specific celltype | specific celltype with
# the dominant bulk filtered out. Vocab is tissue-dispatched via _markers
# (scripts/tissue_markers/{pancreas,thymus}.py) so the panel-3 title and
# exclusion set are biologically meaningful for whichever tissue is active.
# 2026-05-22: replaced hardcoded pancreas dicts here with _markers exports
# so thymus runs no longer produce a meaningless "no exocrine" panel.
LINEAGE_MAP   = dict(_markers.LINEAGE_MAP)
SPECIFIC_KEEP = set(_markers.SPECIFIC_KEEP)
PALETTE       = dict(_markers.PALETTE)
THIRD_EXCLUDE = set(_markers.THIRD_PANEL["exclude_types"])
THIRD_TITLE   = str(_markers.THIRD_PANEL["title"])

mc = adata.obs['manual_celltype'].astype(str)
keep_general  = ~mc.str.startswith('Indeterminate')
keep_specific = mc.isin(SPECIFIC_KEEP)
keep_third    = keep_specific & ~mc.isin(THIRD_EXCLUDE)

adata_g = adata[keep_general.to_numpy()].copy()
adata_g.obs['_lineage'] = pd.Categorical(
    adata_g.obs['manual_celltype'].astype(str).map(LINEAGE_MAP)
)
adata_s = adata[keep_specific.to_numpy()].copy()
adata_s.obs['_specific'] = pd.Categorical(
    adata_s.obs['manual_celltype'].astype(str)
)
adata_n = adata[keep_third.to_numpy()].copy()
adata_n.obs['_specific'] = pd.Categorical(
    adata_n.obs['manual_celltype'].astype(str)
)

# Plot rare types on top: reorder categories most-frequent-first so the last
# scatter draw covers the dense bulk with the rare populations.
def _ordered(cats, counts):
    return sorted(cats, key=lambda c: -counts.get(c, 0))

g_counts = adata_g.obs['_lineage'].value_counts().to_dict()
s_counts = adata_s.obs['_specific'].value_counts().to_dict()
n_counts = adata_n.obs['_specific'].value_counts().to_dict()
adata_g.obs['_lineage'] = adata_g.obs['_lineage'].cat.reorder_categories(
    _ordered(list(adata_g.obs['_lineage'].cat.categories), g_counts), ordered=False
)
adata_s.obs['_specific'] = adata_s.obs['_specific'].cat.reorder_categories(
    _ordered(list(adata_s.obs['_specific'].cat.categories), s_counts), ordered=False
)
adata_n.obs['_specific'] = adata_n.obs['_specific'].cat.reorder_categories(
    _ordered(list(adata_n.obs['_specific'].cat.categories), n_counts), ordered=False
)

fig, axes = plt.subplots(1, 3, figsize=(28, 9))
sc.pl.umap(adata_g, color='_lineage', ax=axes[0], show=False,
           size=3, frameon=False, legend_loc='right margin',
           legend_fontsize=9, palette=PALETTE,
           title=f'General lineage (n={adata_g.n_obs:,})')
sc.pl.umap(adata_s, color='_specific', ax=axes[1], show=False,
           size=3, frameon=False, legend_loc='right margin',
           legend_fontsize=9, palette=PALETTE,
           title=f'Specific celltype (n={adata_s.n_obs:,})')
sc.pl.umap(adata_n, color='_specific', ax=axes[2], show=False,
           size=4, frameon=False, legend_loc='right margin',
           legend_fontsize=9, palette=PALETTE,
           title=f'{THIRD_TITLE} (n={adata_n.n_obs:,})')
for ax in axes:
    for coll in ax.collections:
        coll.set_rasterized(True)
fig.suptitle(f'{SAMPLE_ID} — cluster consensus annotation', y=0.99)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cluster_consensus_annotation.png',
            dpi=300, bbox_inches='tight')
plt.show()

del adata_g, adata_s, adata_n


## 9. Final Cell Type Assignment

In [ ]:
adata.obs['celltype'] = adata.obs['manual_celltype']

print("\nFinal cell type distribution:")
print(adata.obs['celltype'].value_counts())

# Color scaling: clip total_counts and n_genes at p95 so contrast is visible
# across the bulk distribution. Larger legend fonts so categorical labels are
# readable on 1.2M-cell UMAPs.
import numpy as _np
# final_celltypes_overview.png — general lineage (de-crowded) + leiden + QC,
# adaptive point size. Specific-celltype detail lives split in
# cluster_consensus_annotation.png / predicted_celltypes.png. Shares utils
# helpers with scripts/rerender_stage02_celltype_figs.py.
from utils.analysis_utils import umap_dot_size
_lin_map = dict(_markers.LINEAGE_MAP); _pal = dict(_markers.PALETTE)
adata.obs['_lineage'] = pd.Categorical(
    adata.obs['celltype'].astype(str).map(_lin_map).fillna('Other'))
_lin_pal = {c: _pal.get(c, '#cccccc') for c in adata.obs['_lineage'].cat.categories}
_sz = umap_dot_size(adata.n_obs)
fig, axes = plt.subplots(2, 2, figsize=(15, 15))
sc.pl.umap(adata, color='_lineage', ax=axes[0, 0], show=False, size=_sz, palette=_lin_pal,
           frameon=False, legend_loc='right margin', legend_fontsize=10,
           title=f"Cell lineage ({adata.obs['_lineage'].nunique()} groups)")
sc.pl.umap(adata, color='leiden_scvi', ax=axes[0, 1], show=False, size=_sz,
           title='scVI Leiden', legend_loc='on data', legend_fontsize=10, legend_fontoutline=2)
for _col, _ax, _title in [('total_counts', axes[1, 0], 'Total Counts'),
                          ('n_genes_by_counts', axes[1, 1], 'N Genes')]:
    if _col in adata.obs.columns:
        _vmax = float(np.quantile(adata.obs[_col], 0.95))
        sc.pl.umap(adata, color=_col, ax=_ax, show=False, cmap='viridis', size=_sz,
                   title=f"{_title} (clipped p95={_vmax:.0f})",
                   vmin=float(adata.obs[_col].min()), vmax=_vmax)
    else:
        _ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_celltypes_overview.png', dpi=300, bbox_inches='tight')
plt.show()

# ---- Lineage phenotyping: refine multi-lineage cells via canary-marker depth ----
# For each cell, count how many DISTINCT lineage-specific markers each canary
# panel detects (raw count >= 3). Cells with multiple exclusive groups detected
# are resolved to their dominant lineage (top-depth) when the margin >= 2 and
# top_depth >= 2. Cells flagged this way are tagged "lineage_phenotyped".
# Cells with >= 3 mutually-exclusive groups are tagged "doublet_suspected"
# (kept as-is — no resolution attempted).
# See scripts/verify_upper_tail_doublets.py and scripts/lineage_phenotype.py
# for the standalone post-hoc form.

LINEAGE_CANARY_PANELS = _markers.LINEAGE_CANARY_PANELS
LINEAGE_EXCLUSIVE_GROUP = _markers.LINEAGE_EXCLUSIVE_GROUP
# Detection threshold on the LOGNORM layer (sample-comparable across
# samples with different sequencing depth). log1p(3) ≈ 1.386.
LOG_DETECT = float(np.log1p(3.0))
DOMINANCE_MARGIN = 2

if "counts" in adata.layers:
    _gene_idx = {g: i for i, g in enumerate(adata.var_names)}
    _all_canary = sorted({g for genes in LINEAGE_CANARY_PANELS.values() for g in genes})
    _present = [g for g in _all_canary if g in _gene_idx]
    _cols = [_gene_idx[g] for g in _present]
    _pos = {g: j for j, g in enumerate(_present)}
    # Use lognorm layer for sample-comparable thresholding (computed earlier in cell 5).
    _layer = adata.layers["lognorm"] if "lognorm" in adata.layers else adata.layers["counts"]
    _canary = _layer[:, _cols].toarray() if hasattr(_layer, "toarray") else np.asarray(_layer[:, _cols])
    _canary = np.asarray(_canary)

    _lineages = list(LINEAGE_CANARY_PANELS.keys())
    _depth = np.zeros((adata.n_obs, len(_lineages)), dtype=np.int8)
    _sum = np.zeros((adata.n_obs, len(_lineages)), dtype=np.float32)
    for _li, _lin in enumerate(_lineages):
        _idxs = [_pos[g] for g in LINEAGE_CANARY_PANELS[_lin] if g in _pos]
        if not _idxs:
            continue
        _sub = _canary[:, _idxs]
        _depth[:, _li] = (_sub >= LOG_DETECT).sum(axis=1).astype(np.int8)
        _sum[:, _li] = _sub.sum(axis=1)

    _detected = _depth >= 1
    _group_names = sorted(set(LINEAGE_EXCLUSIVE_GROUP[lin] for lin in _lineages))
    _group_to_idx = {g: i for i, g in enumerate(_group_names)}
    _lin_group = np.array([_group_to_idx[LINEAGE_EXCLUSIVE_GROUP[lin]] for lin in _lineages])
    _group_hit = np.zeros((adata.n_obs, len(_group_names)), dtype=bool)
    for _li in range(len(_lineages)):
        if not _detected[:, _li].any():
            continue
        _group_hit[:, _lin_group[_li]] |= _detected[:, _li]
    _n_groups = _group_hit.sum(axis=1).astype(np.int8)

    _composite = _depth.astype(np.float64) * 1000.0 + _sum.astype(np.float64)
    _top = _composite.argmax(axis=1)
    _td = _depth[np.arange(adata.n_obs), _top]
    _composite2 = _composite.copy()
    _composite2[np.arange(adata.n_obs), _top] = -1
    _second = _composite2.argmax(axis=1)
    _sd = _depth[np.arange(adata.n_obs), _second]
    _margin = _td.astype(np.int16) - _sd.astype(np.int16)

    _status = np.empty(adata.n_obs, dtype=object)
    _is_doublet = _n_groups >= 3
    _is_multi = (_n_groups == 2) & ~_is_doublet
    _is_single = _n_groups <= 1
    _status[_is_single] = "single_lineage"
    _status[_is_doublet] = "doublet_suspected"
    _can_resolve = _is_multi & (_margin >= DOMINANCE_MARGIN) & (_td >= 2)
    _status[_is_multi & _can_resolve] = "lineage_phenotyped"
    _status[_is_multi & ~_can_resolve] = "lineage_ambiguous"

    _lin_arr = np.array(_lineages)
    _ct_orig = adata.obs["celltype"].astype(str).to_numpy()
    _ct_lineage = _ct_orig.copy()
    _resolved = _status == "lineage_phenotyped"
    _ct_lineage[_resolved] = _lin_arr[_top[_resolved]]
    _dominant = np.empty(adata.n_obs, dtype=object); _dominant[:] = ""
    _dominant[~_is_single] = _lin_arr[_top[~_is_single]]

    adata.obs["lineage_status"] = pd.Categorical(_status,
        categories=["single_lineage", "lineage_phenotyped",
                    "lineage_ambiguous", "doublet_suspected"])
    adata.obs["celltype_lineage"] = pd.Categorical(_ct_lineage)
    adata.obs["lineage_dominant"] = pd.Categorical(_dominant)
    adata.obs["lineage_top_depth"] = _td.astype(np.int8)
    adata.obs["lineage_second_depth"] = _sd.astype(np.int8)
    adata.obs["lineage_n_groups"] = _n_groups

    print("\n--- Lineage phenotyping ---")
    print(adata.obs["lineage_status"].value_counts().to_string())
    _diff = (_ct_orig != _ct_lineage).sum()
    print(f"  cells reclassified: {int(_diff):,}")
else:
    print("[skipped] lineage phenotyping — counts layer missing.")


## 10. Save Annotated Data

In [ ]:
output_file = OUTPUT_DIR / f"{SAMPLE_ID}_phenotyped.h5ad"
adata.write_h5ad(output_file)

print(f"\nPhenotyped data saved to: {output_file}")
print(f"Dataset shape: {adata.shape}")
print(f"\nCell type annotations:")
print(adata.obs['celltype'].value_counts())

# 2026-05-22: switched from [celltype, predicted_celltype, celltype_confidence]
# to a resolution-clean column set. predicted_celltype carries Indeterminate
# bucket strings which violate project policy (#15: Indeterminate must never
# appear in final data). The CSV is later overwritten by
# scripts/rerender_stage02_celltype_figs.py after resolve_indeterminate runs,
# so this initial export is a stage-02 snapshot — already clean of the worst
# leak (no predicted_celltype column).
_export_cols = ["celltype", "celltype_confidence"]
for _opt in ("celltype_lineage", "celltype_resolution",
             "celltype_neighbor_agreement"):
    if _opt in adata.obs.columns:
        _export_cols.append(_opt)
celltype_export = adata.obs[_export_cols]
celltype_export.to_csv(OUTPUT_DIR / f"{SAMPLE_ID}_celltype_assignments.csv")
print(f"\nCell type assignments exported to: "
      f"{OUTPUT_DIR / f'{SAMPLE_ID}_celltype_assignments.csv'}  "
      f"({len(_export_cols)} cols: {_export_cols})")
print("  Note: this is the stage-02 snapshot. The final CSV (post-resolve) "
      "is overwritten by scripts/rerender_stage02_celltype_figs.py.")


---

## Next Steps

The annotated data is ready for:
1. **Spatial analysis** (03_spatial_analysis.ipynb)
2. **Group-wise comparisons** (04_group_comparisons.ipynb)
3. **Tissue comparisons** (05_tissue_comparisons.ipynb)